In [ ]:
# Method 6 Step 1: Setup (graph construction first, model loading later)

from google.colab import drive
drive.mount('/content/drive')

!pip -q install rank_bm25 spacy
!python -m spacy download en_core_web_sm -q

import re, json, time
from pathlib import Path
from collections import Counter, defaultdict
import spacy

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")

with open(PROJECT_DIR / "hotpot_dev_distractor_v1.json") as f:
    hotpot_dev = json.load(f)
print(f"Dev: {len(hotpot_dev)}")

nlp = spacy.load("en_core_web_sm")
print("spaCy loaded. No model loaded yet — saving GPU for later.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dev: 7405
spaCy loaded. No model loaded yet — saving GPU for later.


In [ ]:
import torch
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

GPU allocated: 41.9 GB
GPU reserved: 41.9 GB


In [ ]:
# Method 6 Step 2: Entity extraction + graph construction

import time
from collections import defaultdict

def hotpot_context_to_text(c):
    title, sents = c
    return f"Title: {title}\n{' '.join(sents)}"

def extract_entities(text):
    """Extract named entities using spaCy."""
    doc = nlp(text[:5000])  # cap length for speed
    entities = set()
    for ent in doc.ents:
        entities.add(ent.text.lower().strip())
    # Also add noun chunks as potential entities
    for chunk in doc.noun_chunks:
        if len(chunk.text.split()) <= 4:
            entities.add(chunk.text.lower().strip())
    return entities

def build_entity_graph(example):
    """
    Build a graph where:
    - Nodes = paragraphs (indexed 0-9)
    - Edges = shared entities between paragraphs
    Returns: paragraph entities, adjacency list with shared entity counts
    """
    paragraphs = example["context"]

    # Extract entities from each paragraph (including title)
    para_entities = []
    for ctx in paragraphs:
        title, sents = ctx
        full_text = title + ". " + " ".join(sents)
        ents = extract_entities(full_text)
        # Always add the title as an entity
        ents.add(title.lower().strip())
        para_entities.append(ents)

    # Build adjacency: paragraphs sharing entities
    adjacency = defaultdict(list)
    for i in range(len(paragraphs)):
        for j in range(i + 1, len(paragraphs)):
            shared = para_entities[i] & para_entities[j]
            if shared:
                adjacency[i].append((j, len(shared), shared))
                adjacency[j].append((i, len(shared), shared))

    return para_entities, adjacency

def graph_retrieve(example, k=2):
    """
    Graph-based retrieval:
    1. Extract entities from the question
    2. Find paragraphs that share entities with the question (hop 1)
    3. From hop 1 paragraphs, follow edges to connected paragraphs (hop 2)
    4. Score paths by entity overlap and return top-k paragraphs
    """
    question = example["question"]
    paragraphs = example["context"]

    # Extract question entities
    q_entities = extract_entities(question)
    # Also add individual words as fallback matching
    q_words = set(question.lower().split())

    # Build graph
    para_entities, adjacency = build_entity_graph(example)

    # Score each paragraph by question entity overlap
    para_scores = []
    for i, ents in enumerate(para_entities):
        # Entity match score
        entity_overlap = len(q_entities & ents)
        # Title word overlap (titles are important in HotpotQA)
        title = paragraphs[i][0].lower()
        title_words = set(title.split())
        title_overlap = len(q_words & title_words)
        # Combined score
        score = entity_overlap * 2 + title_overlap * 3
        para_scores.append((i, score))

    # Sort by score — top paragraphs are hop 1 candidates
    para_scores.sort(key=lambda x: x[1], reverse=True)

    # Strategy: pick best hop-1 paragraph, then find its best connected neighbor
    selected = set()
    selected_list = []

    # Try each hop-1 candidate
    for hop1_idx, hop1_score in para_scores:
        if hop1_score == 0:
            continue

        # Find best hop-2 neighbor not already selected
        best_hop2 = None
        best_hop2_score = -1

        for neighbor_idx, shared_count, shared_ents in adjacency.get(hop1_idx, []):
            if neighbor_idx == hop1_idx:
                continue
            # Score neighbor by: shared entities with hop1 + question overlap
            neighbor_q_overlap = len(q_entities & para_entities[neighbor_idx])
            combined = shared_count + neighbor_q_overlap
            if combined > best_hop2_score:
                best_hop2_score = combined
                best_hop2 = neighbor_idx

        if best_hop2 is not None and len(selected) < k:
            if hop1_idx not in selected:
                selected.add(hop1_idx)
                selected_list.append(hop1_idx)
            if best_hop2 not in selected:
                selected.add(best_hop2)
                selected_list.append(best_hop2)

        if len(selected) >= k:
            break

    # Fallback: if graph didn't find enough, add top BM25-scored paragraphs
    if len(selected) < k:
        for idx, _ in para_scores:
            if idx not in selected:
                selected.add(idx)
                selected_list.append(idx)
            if len(selected) >= k:
                break

    # Build context from selected paragraphs
    context = "\n\n".join(hotpot_context_to_text(paragraphs[i]) for i in selected_list[:k])

    # Calculate retrieval recall
    supporting_titles = set(t for t, _ in example["supporting_facts"])
    retrieved_titles = set(paragraphs[i][0] for i in selected_list[:k])
    recall = len(supporting_titles & retrieved_titles) / len(supporting_titles) if supporting_titles else 0
    full_recall = int(supporting_titles <= retrieved_titles)

    return context, recall, full_recall, selected_list[:k]

def get_qtype(ex):
    return ex.get("type", "unknown")

# Rerun test on 10 examples
print("Testing graph retrieval on 10 examples:\n")
for i, ex in enumerate(hotpot_dev[:10], start=1):
    ctx, recall, full, indices = graph_retrieve(ex, k=2)

    supporting_titles = set(t for t, _ in ex["supporting_facts"])
    retrieved_titles = [ex["context"][j][0] for j in indices]

    print(f"Ex {i} ({get_qtype(ex)}): {ex['question'][:65]}")
    print(f"  Gold: {sorted(supporting_titles)}")
    print(f"  Retrieved: {retrieved_titles}")
    print(f"  Recall: {recall:.1f} | Full: {full}")
    print()

Testing graph retrieval on 10 examples:

Ex 1 (comparison): Were Scott Derrickson and Ed Wood of the same nationality?
  Gold: ['Ed Wood', 'Scott Derrickson']
  Retrieved: ['Ed Wood (film)', 'Conrad Brooks']
  Recall: 0.0 | Full: 0

Ex 2 (bridge): What government position was held by the woman who portrayed Corl
  Gold: ['Kiss and Tell (1945 film)', 'Shirley Temple']
  Retrieved: ['A Kiss for Corliss', 'Kiss and Tell (1945 film)']
  Recall: 0.5 | Full: 0

Ex 3 (bridge): What science fantasy young adult series, told in first person, ha
  Gold: ['Animorphs', 'The Hork-Bajir Chronicles']
  Retrieved: ['List of Square Enix companion books', 'Victoria Hanley']
  Recall: 0.0 | Full: 0

Ex 4 (comparison): Are the Laleli Mosque and Esma Sultan Mansion located in the same
  Gold: ['Esma Sultan Mansion', 'Laleli Mosque']
  Retrieved: ['Esma Sultan Mansion', 'Sultan Ahmed Mosque']
  Recall: 0.5 | Full: 0

Ex 5 (bridge): The director of the romantic comedy "Big Stone Gap" is based in w
  Gold: ['A

In [ ]:
# Method 6 Step 3: Improved graph retrieval — title-centric

def graph_retrieve_v2(example, k=2):
    """
    Improved graph retrieval focused on title matching:
    1. Match question words to paragraph titles (hop 1)
    2. Find entities in hop-1 paragraphs that match other titles (hop 2)
    """
    question = example["question"]
    paragraphs = example["context"]
    q_lower = question.lower()

    # Step 1: Score paragraphs by title presence in question
    title_scores = []
    for i, (title, sents) in enumerate(paragraphs):
        title_lower = title.lower()
        # Check if title (or significant part) appears in question
        title_words = set(title_lower.split())
        q_words = set(q_lower.split())

        # Exact title match in question
        if title_lower in q_lower:
            score = 100
        # Partial title match (at least 2 content words)
        else:
            # Remove common words
            stopwords = {'the', 'a', 'an', 'of', 'in', 'on', 'at', 'to', 'for', 'and', 'is', 'was', 'are', 'were'}
            title_content = title_words - stopwords
            q_content = q_words - stopwords
            overlap = title_content & q_content
            if len(overlap) >= 2:
                score = 50 + len(overlap) * 10
            elif len(overlap) == 1 and len(title_content) <= 2:
                score = 30
            else:
                score = 0

        title_scores.append((i, score))

    title_scores.sort(key=lambda x: x[1], reverse=True)

    # Step 2: For each hop-1 candidate, find which other titles are mentioned in its text
    selected_pairs = []

    for hop1_idx, hop1_score in title_scores:
        if hop1_score == 0:
            continue

        hop1_title, hop1_sents = paragraphs[hop1_idx]
        hop1_text = " ".join(hop1_sents).lower()

        # Check which other paragraph titles appear in hop1's text
        for j, (other_title, _) in enumerate(paragraphs):
            if j == hop1_idx:
                continue
            other_lower = other_title.lower()

            # Check if other title appears in hop1's text
            if other_lower in hop1_text:
                # Also check if this other paragraph mentions something from the question
                other_text = " ".join(paragraphs[j][1]).lower()
                q_ents = extract_entities(question)
                other_ents = extract_entities(other_text + " " + other_title)
                relevance = len(q_ents & other_ents)

                selected_pairs.append((hop1_idx, j, hop1_score + relevance * 5))

        # Also check if hop1's title appears in other paragraphs' text (reverse direction)
        for j, (other_title, other_sents) in enumerate(paragraphs):
            if j == hop1_idx:
                continue
            other_text = " ".join(other_sents).lower()
            if hop1_title.lower() in other_text:
                q_ents = extract_entities(question)
                other_ents = extract_entities(other_text + " " + other_title)
                relevance = len(q_ents & other_ents)
                selected_pairs.append((j, hop1_idx, hop1_score + relevance * 5))

    # Deduplicate and pick the best pair
    seen = set()
    best_pair = None
    best_score = -1

    for a, b, score in selected_pairs:
        pair_key = (min(a, b), max(a, b))
        if pair_key in seen:
            continue
        seen.add(pair_key)
        if score > best_score:
            best_score = score
            best_pair = (a, b)

    # Build result
    if best_pair:
        selected = list(best_pair)
    else:
        # Fallback: top-2 by title score
        selected = [idx for idx, _ in title_scores[:k]]

    context = "\n\n".join(hotpot_context_to_text(paragraphs[i]) for i in selected[:k])

    supporting_titles = set(t for t, _ in example["supporting_facts"])
    retrieved_titles = set(paragraphs[i][0] for i in selected[:k])
    recall = len(supporting_titles & retrieved_titles) / len(supporting_titles) if supporting_titles else 0
    full_recall = int(supporting_titles <= retrieved_titles)

    return context, recall, full_recall, selected[:k]

# Test on same 10 examples
print("Graph v2 retrieval on 10 examples:\n")
for i, ex in enumerate(hotpot_dev[:10], start=1):
    ctx, recall, full, indices = graph_retrieve_v2(ex, k=2)

    supporting_titles = set(t for t, _ in ex["supporting_facts"])
    retrieved_titles = [ex["context"][j][0] for j in indices]

    print(f"Ex {i} ({get_qtype(ex)}): {ex['question'][:65]}")
    print(f"  Gold: {sorted(supporting_titles)}")
    print(f"  Retrieved: {retrieved_titles}")
    print(f"  Recall: {recall:.1f} | Full: {full}")
    print()

Graph v2 retrieval on 10 examples:

Ex 1 (comparison): Were Scott Derrickson and Ed Wood of the same nationality?
  Gold: ['Ed Wood', 'Scott Derrickson']
  Retrieved: ['Tyler Bates', 'Scott Derrickson']
  Recall: 0.5 | Full: 0

Ex 2 (bridge): What government position was held by the woman who portrayed Corl
  Gold: ['Kiss and Tell (1945 film)', 'Shirley Temple']
  Retrieved: ['Charles Craft', 'Meet Corliss Archer']
  Recall: 0.0 | Full: 0

Ex 3 (bridge): What science fantasy young adult series, told in first person, ha
  Gold: ['Animorphs', 'The Hork-Bajir Chronicles']
  Retrieved: ['List of Square Enix companion books', 'Science Fantasy (magazine)']
  Recall: 0.0 | Full: 0

Ex 4 (comparison): Are the Laleli Mosque and Esma Sultan Mansion located in the same
  Gold: ['Esma Sultan Mansion', 'Laleli Mosque']
  Retrieved: ['Esma Sultan Mansion', 'Esma Sultan']
  Recall: 0.5 | Full: 0

Ex 5 (bridge): The director of the romantic comedy "Big Stone Gap" is based in w
  Gold: ['Adriana Trigia

In [ ]:
# Missing helpers

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pt = normalize_text(pred).split()
    gt = normalize_text(gold).split()
    common = Counter(pt) & Counter(gt)
    ns = sum(common.values())
    if not pt or not gt: return int(pt == gt)
    if ns == 0: return 0.0
    p, r = ns/len(pt), ns/len(gt)
    return 2*p*r/(p+r)

print("Helpers defined.")

Helpers defined.


In [ ]:
# Method 6 Step 4: Hybrid BM25 + Graph retrieval

from rank_bm25 import BM25Okapi

def hybrid_graph_retrieve(example, k=2):
    """
    Hybrid approach:
    1. BM25 finds the best hop-1 paragraph
    2. Entity graph finds the connected hop-2 paragraph
    """
    question = example["question"]
    paragraphs = example["context"]


    # Step 1: BM25 to find hop-1 candidates
    para_texts = [hotpot_context_to_text(c) for c in paragraphs]
    tokenized = [simple_tokenize(t) for t in para_texts]
    bm25 = BM25Okapi(tokenized)
    q_tokens = simple_tokenize(question)
    bm25_scores = bm25.get_scores(q_tokens)

    bm25_ranked = sorted(range(len(bm25_scores)),
                         key=lambda i: bm25_scores[i], reverse=True)

    # Step 2: For top BM25 results, use entity/title graph to find hop-2
    best_pair = None
    best_pair_score = -1

    for hop1_idx in bm25_ranked[:5]:  # check top-5 BM25 results as hop-1
        hop1_title = paragraphs[hop1_idx][0]
        hop1_text = " ".join(paragraphs[hop1_idx][1]).lower()
        hop1_full = (hop1_title + " " + hop1_text).lower()

        # Extract entities from hop-1 paragraph
        hop1_entities = extract_entities(hop1_full)

        for j, (other_title, other_sents) in enumerate(paragraphs):
            if j == hop1_idx:
                continue

            other_text = " ".join(other_sents).lower()
            other_full = (other_title + " " + other_text).lower()
            other_title_lower = other_title.lower()

            # Connection score between hop1 and candidate hop2
            connection_score = 0

            # Strong signal: hop2's title appears in hop1's text
            if other_title_lower in hop1_full:
                connection_score += 50

            # Strong signal: hop1's title appears in hop2's text
            if hop1_title.lower() in other_full:
                connection_score += 50

            # Medium signal: shared entities
            other_entities = extract_entities(other_full)
            shared = hop1_entities & other_entities
            # Remove very common words from shared
            shared = {e for e in shared if len(e) > 3}
            connection_score += len(shared) * 2

            # Bonus: hop2 also has question relevance
            q_overlap = len(extract_entities(question) & other_entities)
            connection_score += q_overlap * 3

            if connection_score > 0:
                # Total score = BM25 score of hop1 + connection to hop2
                total = bm25_scores[hop1_idx] + connection_score
                if total > best_pair_score:
                    best_pair_score = total
                    best_pair = (hop1_idx, j)

    # Use best pair, or fall back to BM25 top-2
    if best_pair and best_pair_score > 0:
        selected = list(best_pair)
    else:
        selected = bm25_ranked[:k]

    context = "\n\n".join(hotpot_context_to_text(paragraphs[i]) for i in selected[:k])

    supporting_titles = set(t for t, _ in example["supporting_facts"])
    retrieved_titles = set(paragraphs[i][0] for i in selected[:k])
    recall = len(supporting_titles & retrieved_titles) / len(supporting_titles) if supporting_titles else 0
    full_recall = int(supporting_titles <= retrieved_titles)

    return context, recall, full_recall, selected[:k]

# Test on 10 examples + compare with BM25
print("Hybrid Graph vs BM25 on 10 examples:\n")

graph_full = 0
bm25_full = 0

for i, ex in enumerate(hotpot_dev[:10], start=1):
    # Graph
    g_ctx, g_recall, g_full, g_idx = hybrid_graph_retrieve(ex, k=2)
    graph_full += g_full

    # BM25 top-2 for comparison
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    bm25_ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:2]
    supp = set(t for t, _ in ex["supporting_facts"])
    bm25_ret = set(ex["context"][j][0] for j in bm25_ranked)
    b_full = int(supp <= bm25_ret)
    bm25_full += b_full

    g_titles = [ex["context"][j][0] for j in g_idx]
    b_titles = [ex["context"][j][0] for j in bm25_ranked]

    marker = "✓" if g_full else "✗"
    b_marker = "✓" if b_full else "✗"

    print(f"Ex {i} ({get_qtype(ex)}): {ex['question'][:60]}")
    print(f"  Gold:  {sorted(supp)}")
    print(f"  Graph: {g_titles} {marker}")
    print(f"  BM25:  {b_titles} {b_marker}")
    print()

print(f"Full recall: Graph={graph_full}/10, BM25={bm25_full}/10")

Hybrid Graph vs BM25 on 10 examples:

Ex 1 (comparison): Were Scott Derrickson and Ed Wood of the same nationality?
  Gold:  ['Ed Wood', 'Scott Derrickson']
  Graph: ['Scott Derrickson', 'Adam Collis'] ✗
  BM25:  ['Doctor Strange (2016 film)', 'Woodson, Arkansas'] ✗

Ex 2 (bridge): What government position was held by the woman who portrayed
  Gold:  ['Kiss and Tell (1945 film)', 'Shirley Temple']
  Graph: ['A Kiss for Corliss', 'Shirley Temple'] ✗
  BM25:  ['A Kiss for Corliss', 'Kiss and Tell (1945 film)'] ✗

Ex 3 (bridge): What science fantasy young adult series, told in first perso
  Gold:  ['Animorphs', 'The Hork-Bajir Chronicles']
  Graph: ['The Hork-Bajir Chronicles', 'Animorphs'] ✓
  BM25:  ['Animorphs', 'The Hork-Bajir Chronicles'] ✓

Ex 4 (comparison): Are the Laleli Mosque and Esma Sultan Mansion located in the
  Gold:  ['Esma Sultan Mansion', 'Laleli Mosque']
  Graph: ['Esma Sultan Mansion', 'Esma Sultan'] ✗
  BM25:  ['Esma Sultan Mansion', 'Laleli Mosque'] ✓

Ex 5 (bridge)

In [ ]:
# Method 6: Graph vs BM25 retrieval recall — 500 examples

import time

sample = hotpot_dev[:500]
graph_full, bm25_full = 0, 0
graph_partial, bm25_partial = [], []

start = time.time()

for i, ex in enumerate(sample, start=1):
    # Graph retrieval
    _, g_recall, g_full, _ = hybrid_graph_retrieve(ex, k=2)
    graph_full += g_full
    graph_partial.append(g_recall)

    # BM25 top-2
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    bm25_ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:2]
    supp = set(t for t, _ in ex["supporting_facts"])
    bm25_ret = set(ex["context"][j][0] for j in bm25_ranked)
    b_recall = len(supp & bm25_ret) / len(supp) if supp else 0
    bm25_full += int(supp <= bm25_ret)
    bm25_partial.append(b_recall)

    if i % 100 == 0:
        elapsed = time.time() - start
        print(f"[{i}/500] Graph full={graph_full}/{i} ({graph_full/i*100:.1f}%) "
              f"BM25 full={bm25_full}/{i} ({bm25_full/i*100:.1f}%) "
              f"Time={elapsed:.0f}s")

print(f"\n{'='*60}")
print(f"{'Method':<20} {'Full Recall':>15} {'Partial Recall':>15}")
print(f"{'='*60}")
print(f"{'Graph (hybrid)':<20} {graph_full/500*100:>14.1f}% {sum(graph_partial)/500*100:>14.1f}%")
print(f"{'BM25 top-2':<20} {bm25_full/500*100:>14.1f}% {sum(bm25_partial)/500*100:>14.1f}%")
print(f"{'='*60}")

[100/500] Graph full=40/100 (40.0%) BM25 full=35/100 (35.0%) Time=119s
[200/500] Graph full=79/200 (39.5%) BM25 full=70/200 (35.0%) Time=238s
[300/500] Graph full=126/300 (42.0%) BM25 full=100/300 (33.3%) Time=357s
[400/500] Graph full=177/400 (44.2%) BM25 full=132/400 (33.0%) Time=476s
[500/500] Graph full=222/500 (44.4%) BM25 full=159/500 (31.8%) Time=594s

Method                   Full Recall  Partial Recall
Graph (hybrid)                 44.4%           62.5%
BM25 top-2                     31.8%           61.7%


In [ ]:
# Method 6: Full evaluation — Graph vs BM25 retrieval with XXL reader

import time

total = len(hotpot_dev)

em_graph = defaultdict(list)
f1_graph = defaultdict(list)
em_bm25 = defaultdict(list)
f1_bm25 = defaultdict(list)
graph_full_recall = 0
bm25_full_recall = 0

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    q = ex["question"]
    gold = ex["answer"]

    # Graph retrieval + XXL reader
    g_ctx, g_recall, g_full, _ = hybrid_graph_retrieve(ex, k=2)
    g_pred = run_model(build_qa_prompt(q, g_ctx))
    em_graph[qtype].append(compute_em(g_pred, gold))
    f1_graph[qtype].append(compute_f1(g_pred, gold))
    graph_full_recall += g_full

    # BM25 top-2 + XXL reader (same k=2 for fair comparison)
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    scores = bm25.get_scores(simple_tokenize(q))
    bm25_ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:2]
    b_ctx = "\n\n".join(hotpot_context_to_text(ex["context"][j]) for j in bm25_ranked)
    b_pred = run_model(build_qa_prompt(q, b_ctx))
    em_bm25[qtype].append(compute_em(b_pred, gold))
    f1_bm25[qtype].append(compute_f1(b_pred, gold))
    supp = set(t for t, _ in ex["supporting_facts"])
    bm25_full_recall += int(supp <= set(ex["context"][j][0] for j in bm25_ranked))

    if i % 500 == 0:
        elapsed = time.time() - start
        all_g = [s for q in em_graph for s in em_graph[q]]
        all_b = [s for q in em_bm25 for s in em_bm25[q]]
        print(f"[{i}/{total}] Graph EM={sum(all_g)/len(all_g):.3f} "
              f"BM25 EM={sum(all_b)/len(all_b):.3f} "
              f"G_recall={graph_full_recall/i:.3f} "
              f"B_recall={bm25_full_recall/i:.3f} "
              f"Time={elapsed:.0f}s")

# Final results
print("\n" + "=" * 90)
print(f"{'Method':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12} {'Full Rec.':>12}")
print("=" * 90)

for name, em_d, f1_d, fr in [
    ("M6: Graph + XXL", em_graph, f1_graph, graph_full_recall),
    ("BM25 top-2 + XXL", em_bm25, f1_bm25, bm25_full_recall)
]:
    all_em = [s for q in em_d for s in em_d[q]]
    all_f1 = [s for q in f1_d for s in f1_d[q]]
    bridge = sum(em_d["bridge"]) / len(em_d["bridge"])
    comp = sum(em_d["comparison"]) / len(em_d["comparison"])
    print(f"{name:<25} {bridge:>12.3f} {comp:>12.3f} "
          f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f} "
          f"{fr/total:>12.3f}")

print("=" * 90)
print(f"\nComparison — B2 BM25 top-5 XXL: EM=0.413, F1=0.564")
print(f"Comparison — B3 Oracle XXL:     EM=0.548, F1=0.741")

[500/7405] Graph EM=0.412 BM25 EM=0.360 G_recall=0.444 B_recall=0.318 Time=932s
[1000/7405] Graph EM=0.409 BM25 EM=0.365 G_recall=0.442 B_recall=0.312 Time=1858s
[1500/7405] Graph EM=0.403 BM25 EM=0.366 G_recall=0.431 B_recall=0.305 Time=2790s
[2000/7405] Graph EM=0.404 BM25 EM=0.363 G_recall=0.427 B_recall=0.290 Time=3722s
[2500/7405] Graph EM=0.406 BM25 EM=0.366 G_recall=0.417 B_recall=0.284 Time=4655s
[3000/7405] Graph EM=0.404 BM25 EM=0.367 G_recall=0.410 B_recall=0.283 Time=5601s
[3500/7405] Graph EM=0.405 BM25 EM=0.364 G_recall=0.409 B_recall=0.288 Time=6526s
[4000/7405] Graph EM=0.402 BM25 EM=0.362 G_recall=0.406 B_recall=0.287 Time=7450s
[4500/7405] Graph EM=0.402 BM25 EM=0.362 G_recall=0.408 B_recall=0.285 Time=8389s
[5000/7405] Graph EM=0.402 BM25 EM=0.359 G_recall=0.409 B_recall=0.284 Time=9320s
[5500/7405] Graph EM=0.400 BM25 EM=0.359 G_recall=0.410 B_recall=0.283 Time=10246s
[6000/7405] Graph EM=0.399 BM25 EM=0.358 G_recall=0.411 B_recall=0.284 Time=11179s
[6500/7405] Grap